In [1]:
import numpy as np
import os
EXP_DIR = "/home/zhaopp/solar-energy/results/chronos2/SKIPPD_1024_1_Chronos2_freq15min_q102030405060708090_zero_shot_Exp_0"  # 改成你的实验目录

yq_path = os.path.join(EXP_DIR, "y_quantile.npy")
y_quantile = np.load(yq_path)

yp_path = os.path.join(EXP_DIR, "y_pred.npy")
y_pred = np.load(yp_path)

yt_path = os.path.join(EXP_DIR, "y_true.npy")
y_true = np.load(yt_path)
# 查看文件形状
print(f"y_quantile shape: {y_quantile.shape}")
print(f"y_pred shape: {y_pred.shape}")
print(f"y_true shape: {y_true.shape}")

# 查看前2个样本（前2行）
print("前2个样本：")
print(y_quantile[:2])
print(y_pred[:2])
print(y_true[:2])

print("\n第一个样本的第一个时间步的所有分位数：")
print(y_quantile[0, 0, :])

y_quantile shape: (9488, 1, 9)
y_pred shape: (9488, 1, 1)
y_true shape: (9488, 1, 1)
前2个样本：
[[[-0.11606264 -0.06415415 -0.05052519 -0.00508451  0.038486
    0.02967501  0.09250832  0.09188461  0.1759758 ]]

 [[-0.10861206 -0.05516148 -0.03921843  0.00790596  0.05357742
    0.04069996  0.10169268  0.09402084  0.17252493]]]
[[[0.038486  ]]

 [[0.05357742]]]
[[[-0.08585385]]

 [[-0.08669834]]]

第一个样本的第一个时间步的所有分位数：
[-0.11606264 -0.06415415 -0.05052519 -0.00508451  0.038486    0.02967501
  0.09250832  0.09188461  0.1759758 ]


In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np

CAP = 30.1
EXP_DIR = "/home/zhaopp/solar-energy/results/chronos2/SKIPPD_1024_288_Chronos2_freq15min_q102030405060708090_zero_shot_Exp_0"  # 改成你的实验目录
OUT_FILE = "quantile_metrics.txt"
QUANTILES = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90] 


def mae_rmse(y_true: np.ndarray, y_pred: np.ndarray):
    yt = y_true.reshape(-1).astype(np.float64)
    yp = y_pred.reshape(-1).astype(np.float64)
    mae = float(np.mean(np.abs(yp - yt)))
    rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
    return mae, rmse


def main():
    y_true_path = os.path.join(EXP_DIR, "y_true.npy")
    yq_path = os.path.join(EXP_DIR, "y_quantile.npy")

    if not os.path.exists(y_true_path):
        raise FileNotFoundError(y_true_path)
    if not os.path.exists(yq_path):
        raise FileNotFoundError(yq_path)

    y_true = np.load(y_true_path)      # [N, pred_len, 1] or [N, pred_len]
    y_quantile = np.load(yq_path)      # [N, pred_len, Q] = [N, 1, 10]

    if y_true.ndim == 3 and y_true.shape[-1] == 1:
        y_true_2d = y_true[:, :, 0]
    elif y_true.ndim == 2:
        y_true_2d = y_true
    else:
        raise ValueError(f"Unexpected y_true shape: {y_true.shape}")

    if y_quantile.ndim != 3:
        raise ValueError(f"Unexpected y_quantile shape: {y_quantile.shape} (expect [N, pred_len, Q])")

    N, H, Q = y_quantile.shape
    if len(QUANTILES) != Q:
        raise ValueError(f"QUANTILES length={len(QUANTILES)} != Q={Q}. QUANTILES={QUANTILES}, y_quantile.shape={y_quantile.shape}")

    lines = []
    lines.append("quantile\tMAE\tRMSE\tacc_mae\tacc_rmse\tcap\tN\tpred_len\n")

    best_mae = None  # (mae, q, rmse, acc_mae, acc_rmse)
    best_rmse = None # (rmse, q, mae, acc_mae, acc_rmse)

    for k, q in enumerate(QUANTILES):
        yq = y_quantile[:, :, k]  # [N, H]
        mae, rmse = mae_rmse(y_true_2d, yq)
        acc_mae = 1.0 - mae / CAP
        acc_rmse = 1.0 - rmse / CAP
        lines.append(f"{q:.4f}\t{mae:.6f}\t{rmse:.6f}\t{acc_mae:.6f}\t{acc_rmse:.6f}\t{CAP}\t{N}\t{H}\n")

        if best_mae is None or mae < best_mae[0]:
            best_mae = (mae, q, rmse, acc_mae, acc_rmse)
        if best_rmse is None or rmse < best_rmse[0]:
            best_rmse = (rmse, q, mae, acc_mae, acc_rmse)

    out_path = os.path.join(EXP_DIR, OUT_FILE)
    with open(out_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

    print(f"Saved: {out_path}")
    print(f"y_true: {y_true.shape}  y_quantile: {y_quantile.shape}")
    print(f"Best MAE : q={best_mae[1]} MAE={best_mae[0]:.6f} RMSE={best_mae[2]:.6f} acc_mae={best_mae[3]:.6f} acc_rmse={best_mae[4]:.6f}")
    print(f"Best RMSE: q={best_rmse[1]} RMSE={best_rmse[0]:.6f} MAE={best_rmse[2]:.6f} acc_mae={best_rmse[3]:.6f} acc_rmse={best_rmse[4]:.6f}")


if __name__ == "__main__":
    main()


Saved: /home/zhaopp/solar-energy/results/chronos2/SKIPPD_1024_288_Chronos2_freq15min_q102030405060708090_zero_shot_Exp_0/quantile_metrics.txt
y_true: (9201, 288, 1)  y_quantile: (9201, 288, 9)
Best MAE : q=0.5 MAE=1.274279 RMSE=3.226874 acc_mae=0.957665 acc_rmse=0.892795
Best RMSE: q=0.4 RMSE=3.201241 MAE=1.292424 acc_mae=0.957062 acc_rmse=0.893646


In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np

CAP = 30.1
EXP_DIR = "/home/zhaopp/solar-energy/results/chronos2/SKIPPD_1024_288_Chronos2_freq15min_q152535455565758595_zero_shot_Exp_0"  # 改成你的实验目录
OUT_FILE = "quantile_metrics.txt"
QUANTILES = [0.15,0.25,0.35,0.45,0.55,0.65,0.75,0.85,0.95] 


def mae_rmse(y_true: np.ndarray, y_pred: np.ndarray):
    yt = y_true.reshape(-1).astype(np.float64)
    yp = y_pred.reshape(-1).astype(np.float64)
    mae = float(np.mean(np.abs(yp - yt)))
    rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
    return mae, rmse


def main():
    y_true_path = os.path.join(EXP_DIR, "y_true.npy")
    yq_path = os.path.join(EXP_DIR, "y_quantile.npy")

    if not os.path.exists(y_true_path):
        raise FileNotFoundError(y_true_path)
    if not os.path.exists(yq_path):
        raise FileNotFoundError(yq_path)

    y_true = np.load(y_true_path)      # [N, pred_len, 1] or [N, pred_len]
    y_quantile = np.load(yq_path)      # [N, pred_len, Q] = [N, 1, 10]

    if y_true.ndim == 3 and y_true.shape[-1] == 1:
        y_true_2d = y_true[:, :, 0]
    elif y_true.ndim == 2:
        y_true_2d = y_true
    else:
        raise ValueError(f"Unexpected y_true shape: {y_true.shape}")

    if y_quantile.ndim != 3:
        raise ValueError(f"Unexpected y_quantile shape: {y_quantile.shape} (expect [N, pred_len, Q])")

    N, H, Q = y_quantile.shape
    if len(QUANTILES) != Q:
        raise ValueError(f"QUANTILES length={len(QUANTILES)} != Q={Q}. QUANTILES={QUANTILES}, y_quantile.shape={y_quantile.shape}")

    lines = []
    lines.append("quantile\tMAE\tRMSE\tacc_mae\tacc_rmse\tcap\tN\tpred_len\n")

    best_mae = None  # (mae, q, rmse, acc_mae, acc_rmse)
    best_rmse = None # (rmse, q, mae, acc_mae, acc_rmse)

    for k, q in enumerate(QUANTILES):
        yq = y_quantile[:, :, k]  # [N, H]
        mae, rmse = mae_rmse(y_true_2d, yq)
        acc_mae = 1.0 - mae / CAP
        acc_rmse = 1.0 - rmse / CAP
        lines.append(f"{q:.4f}\t{mae:.6f}\t{rmse:.6f}\t{acc_mae:.6f}\t{acc_rmse:.6f}\t{CAP}\t{N}\t{H}\n")

        if best_mae is None or mae < best_mae[0]:
            best_mae = (mae, q, rmse, acc_mae, acc_rmse)
        if best_rmse is None or rmse < best_rmse[0]:
            best_rmse = (rmse, q, mae, acc_mae, acc_rmse)

    out_path = os.path.join(EXP_DIR, OUT_FILE)
    with open(out_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

    print(f"Saved: {out_path}")
    print(f"y_true: {y_true.shape}  y_quantile: {y_quantile.shape}")
    print(f"Best MAE : q={best_mae[1]} MAE={best_mae[0]:.6f} RMSE={best_mae[2]:.6f} acc_mae={best_mae[3]:.6f} acc_rmse={best_mae[4]:.6f}")
    print(f"Best RMSE: q={best_rmse[1]} RMSE={best_rmse[0]:.6f} MAE={best_rmse[2]:.6f} acc_mae={best_rmse[3]:.6f} acc_rmse={best_rmse[4]:.6f}")


if __name__ == "__main__":
    main()


Saved: /home/zhaopp/solar-energy/results/chronos2/SKIPPD_1024_288_Chronos2_freq15min_q152535455565758595_zero_shot_Exp_0/quantile_metrics.txt
y_true: (9201, 288, 1)  y_quantile: (9201, 288, 9)
Best MAE : q=0.45 MAE=1.274298 RMSE=3.203743 acc_mae=0.957665 acc_rmse=0.893563
Best RMSE: q=0.45 RMSE=3.203743 MAE=1.274298 acc_mae=0.957665 acc_rmse=0.893563
